# MACE from first principles

A Pass 3 derivation-and-implementation notebook.

**Target reader:** a machine-learning practitioner who understands tensors, linear layers, gradients, and ordinary message passing, but may know nothing about geometric deep learning.

**Authorship rule:** write every explanation and implementation yourself. This notebook supplies the sequence, questions, shape contracts, visual exercises, and mathematical checks.

# Imports

In [129]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

## Learning route

For every topic: write plain English → connect it to familiar ML → record shapes → code a tiny example → graph it → test the mathematical identity → draft the blog paragraph.

- [ ] atomic geometry and symmetry
- [ ] rotations, invariance, and equivariance
- [ ] Cartesian tensors and their rotation laws
- [ ] decomposition into independent rotational blocks
- [ ] representations, irreps, `l`, and `m`
- [ ] spherical harmonics as Fourier-like modes on the sphere
- [ ] Wigner-D matrices and numerical equivariance
- [ ] channels versus angular components
- [ ] tensor products and Clebsch–Gordan decomposition
- [ ] higher-order coupling, `ν`, and `η_ν`
- [ ] radial basis functions and cutoffs
- [ ] MACE Eqs. 8–11
- [ ] one minimal message block and symmetry tests

**Pedagogical reference:** Sections 5.1–5.3 of *A Hitchhiker's Guide to Geometric GNNs for 3D Atomic Systems*. The prompts also preserve useful questions and analogies from your Claude discussion, but the paper and numerical checks remain the authority.

## 1. Notation ledger — write this yourself

Create a table with one row for each symbol below. Leave no symbol undefined in your eventual blog explanation.

`i`, `j`, `k`, `k̃`, `l`, `m`, `ξ`, `ν`, `L`, `M`, `η_ν`

| Symbol | Meaning |
|---|---|
| `i` | Central / receiving atom |
| `j` | Neighbor atom of `i` |
| `k` | Output feature channel of `A` or `B` |
| `k̃` | Input feature channel being mixed into `k` |
| `l` | Angular-momentum / irrep order |
| `m` | Component within order `l`; `m = -l, ..., +l` |
| `ξ` | Tensor-product factor index; one of the `ν` factors |
| `ν` | Number of `A` factors in the tensor product |
| `L` | Output angular-momentum / irrep order of `B` |
| `M` | Component within output order `L` |
| `η_ν` | Label for the coupling path that produces total order `L` |

### Index warm-up — your explanation

Answer these in your own words:

- If `l = 1`, which `m` values exist?
- For `ν = 2`, what are the two factor slots?
- What does `A[i, k̃, l_ξ, m_ξ]` select?

Blog sentence draft:

For `l = 1`, `m` takes the integer values from `-l` to `+l`: `m = -1, 0, +1`.

For `ν = 2`, there are two tensor-product factor slots: `ξ = 1` and `ξ = 2`. They contain the angular-index pairs `(l₁, m₁)` and `(l₂, m₂)`.

The entry `A[i, k̃, l_ξ, m_ξ]` selects one feature value: the feature at atom `i`, input channel `k̃`, angular order `l_ξ`, and component `m_ξ`.

## Authoring contract for a non-GDL reader

Before continuing, write down the standard every section must meet:

- define a term before abbreviating it;
- give one concrete geometric example before formal notation;
- connect the operation to something familiar from ML;
- state every tensor shape and what each axis means;
- show one visual example whenever the object has geometric meaning;
- test the claimed symmetry numerically;
- state the real/complex basis and normalization convention.

Your reader promise:

# Part I — geometric building blocks

The paper equations come later. First make each primitive visible and testable.

## 2. Atomic geometry

### Your explanation

Explain central atom `i`, neighbor `j`, relative displacement, distance, unit direction, translation invariance, and permutation-invariant neighbor pooling.

### ML bridge

Connect neighbor pooling to aggregation in an ordinary message-passing network.

### Shape ledger

### Blog paragraph

We consider a central carbon atom which has four hydrogen atoms. 

### Ideal tetrahedral coordinates for methane

We place the carbon atom at the origin and choose four hydrogen directions from alternating corners of a cube:

$$
(1,1,1),\quad
(1,-1,-1),\quad
(-1,1,-1),\quad
(-1,-1,1).
$$

To turn these into unit directions, divide each vector by its length:

$$
\hat{\mathbf r}
=
\frac{(\pm1,\pm1,\pm1)}{\sqrt{3}}.
$$

Any pair of these directions has dot product

$$
\hat{\mathbf r}_a \cdot \hat{\mathbf r}_b = -\frac{1}{3}.
$$

Therefore, the angle between any two C–H bonds is

$$
\theta
=
\cos^{-1}\left(-\frac{1}{3}\right)
\approx 109.47^\circ,
$$

which is the tetrahedral bond angle.

These are ideal synthetic coordinates for the geometry lab, not measured experimental coordinates.

In [130]:
pos_C = (torch.tensor((0, 0, 0))).unsqueeze(0)
sqrt_3 = torch.sqrt(torch.tensor(3))
pos_H1 = (torch.tensor((1,1,1))/sqrt_3).unsqueeze(0)
pos_H2 = (torch.tensor((1,-1,-1))/sqrt_3).unsqueeze(0)
pos_H3 = (torch.tensor((-1,1,-1))/sqrt_3).unsqueeze(0)
pos_H4 = (torch.tensor((-1,-1,1))/sqrt_3).unsqueeze(0)
pos_H2.shape
pos_C.shape
positions = torch.concat([pos_C, pos_H1, pos_H2, pos_H3, pos_H4], dim=0)

In [131]:
positions

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])

In [132]:
# Reusable interactive 3D molecular plot
from IPython.display import HTML, display

def plot_molecule_3d(positions, labels, title="Molecule geometry"):
    # Plot atoms and bonds from a central atom to every neighbor.
    xyz = positions.detach().cpu().numpy()
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("positions must have shape (number_of_atoms, 3)")
    if len(labels) != len(xyz):
        raise ValueError("labels must contain one entry per atom")

    fig = go.Figure()
    for neighbor_idx in range(1, len(xyz)):
        fig.add_trace(go.Scatter3d(
            x=[xyz[0, 0], xyz[neighbor_idx, 0]],
            y=[xyz[0, 1], xyz[neighbor_idx, 1]],
            z=[xyz[0, 2], xyz[neighbor_idx, 2]],
            mode="lines",
            line=dict(color="gray", width=6),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers+text",
        text=labels,
        textposition="top center",
        hovertemplate="%{text}<br>(%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>",
        marker=dict(
            size=[22] + [14] * (len(xyz) - 1),
            color=["black"] + ["lightgray"] * (len(xyz) - 1),
            line=dict(color="darkslategray", width=1),
        ),
        showlegend=False,
    ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=45),
    )
    return fig

fig = plot_molecule_3d(
    positions,
    labels=["C", "H1", "H2", "H3", "H4"],
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [133]:
torch.manual_seed(42)
translation_vector = 5*torch.rand(1, 3)
new_positions = positions + translation_vector
neighbor_order = torch.randperm(4) + 1
indices = torch.cat([torch.tensor([0]), neighbor_order])
new_pos_perm = new_positions[indices]
labels=["C", "H1", "H2", "H3", "H4"]
new_labels_perm = [labels[i] for i in indices.tolist()]
# new_pos_perm
fig = plot_molecule_3d(
    new_pos_perm,
    labels=new_labels_perm,
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

When we rotate the entire molecule, every position vector changes direction, but all pairwise distances remain unchanged; distances are rotationally invariant.

In [134]:
print("original order:", labels)
print("permuted order:", new_labels_perm)
print("row indices:", indices.tolist())

original order: ['C', 'H1', 'H2', 'H3', 'H4']
permuted order: ['C', 'H3', 'H2', 'H4', 'H1']
row indices: [0, 3, 2, 4, 1]


In [135]:
print(positions - positions[0])


tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])


In [136]:
new_positions - new_positions[0]

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])

In [137]:
# LAB 2 — atomic geometry
# TODO (you): create one central atom and at least three neighbors in 3D.
# TODO (you): compute relative displacements, distances, and unit directions.
# TODO (you): make a labeled 3D scatter/quiver plot.
# TODO (you): translate all atoms and permute neighbor order.
# PASS: relative geometry survives translation and pooled output survives permutation.

## 3. Rotations, invariance, and equivariance

### Your explanation

Using one rotated vector, explain a rotation matrix, an invariant quantity, an equivariant quantity, and the two routes in an equivariance check.

### ML bridge

Compare equivariance with weight sharing in convolution, without claiming they are identical.

### Blog paragraph

3D rotation matrices are basically numbers which help us rotate a vector around an axis. Let us use a simple rotation to demonstrate, such as a 90° rotation on the Z-axis. 

The rotation matrix for this is of the form
[
    cos(theta) 0 0
    0 sin(theta) 0
    0 0 1
]

In [138]:
positions

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])

In [139]:
torch.tensor([[1,2],[3,4]])

tensor([[1, 2],
        [3, 4]])

In [140]:
def rotate_z_axis(positions, theta):
    theta = torch.tensor(theta)
    c = torch.cos(theta)
    s = torch.sin(theta)
    rotation_matrix = torch.tensor(
        [
            [c, -s, 0],
            [s, c, 0],
            [0,0,1]
        ]
        )
    rotated_positions = positions @ rotation_matrix.T 
    return rotation_matrix, rotated_positions
rotation_matrix, rotated_positions = rotate_z_axis(positions, math.pi/2)

In [141]:
print(positions)
print(rotated_positions)

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.5774,  0.5774,  0.5774],
        [ 0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774, -0.5774],
        [ 0.5774, -0.5774,  0.5774]])


In [142]:
fig = plot_molecule_3d(
    rotated_positions,
    labels=new_labels_perm,
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [143]:
print(rotation_matrix.T @ rotation_matrix)
print(torch.linalg.det(rotation_matrix))

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor(1.)


R transpose R = identity, which means the rotation preserves dot products. Preserving dot products means that lengths and angles are preserved, so R transpose is the inverse rotation, and R is these two lengths and vectors. 

In [ ]:
rot_vec1, new_position_1 = rotate_z_axis(positions, math.pi/4)
rot_vec2, new_position_2 = rotate_z_axis(new_position_1, math.pi/3)
print(new_position_2)
new_position_3 = positions @ (rot_vec2 @ rot_vec1).T
print(new_position_3)


torch.Size([5, 3])
torch.Size([5, 3])
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.7071,  0.4082,  0.5774],
        [ 0.4082,  0.7071, -0.5774],
        [-0.4082, -0.7071, -0.5774],
        [ 0.7071, -0.4082,  0.5774]])
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.7071,  0.4082,  0.5774],
        [ 0.4082,  0.7071, -0.5774],
        [-0.4082, -0.7071, -0.5774],
        [ 0.7071, -0.4082,  0.5774]])


Again, in the cell above, I have verified that rotating the atom once and then rotating it again, and then doing the composite rotation, amounts to the same thing. As we can see, the two matrices are exactly the same. 

In [ ]:
# LAB 3 — rotations
# TODO (you): construct a valid 3D rotation matrix from an axis and angle.
# TODO (you): plot a toy neighborhood before and after rotation.
# TODO (you): track one invariant scalar and one equivariant vector.
# CHECK: RᵀR = I, det(R) = +1, and vector norms are preserved.
# CHECK: composing two rotations agrees with sequential application.

## 4. Cartesian tensors before irreps

Do not use the word *irrep* yet. Begin with objects an ML practitioner already knows: a scalar, a 3D vector, and a 3×3 matrix.

### Your explanation

Starting with two vectors `v` and `w`, explain the outer product `T = v ⊗ w`, why it has shape `(3, 3)`, and why it contains nine numbers rather than three. Then derive its rotation law from `v' = Rv` and `w' = Rw`.

### Shape and transformation ledger

Record the shape and rotation law of a scalar, a vector, and this matrix. Explicitly distinguish **tensor rank** from the number of stored components.

### ML bridge

Relate the outer product to multiplicative feature interactions, but explain why geometry dictates how the result must transform.

### Blog paragraph

In [ ]:
# LAB 4 — a Cartesian rank-2 tensor
# TODO (you): choose two non-parallel 3D vectors v and w.
# TODO (you): form T = v ⊗ w and print both its 3×3 and flat 9-value forms.
# TODO (you): rotate v and w, then form T_rotated_from_vectors.
# TODO (you): separately transform T using R @ T @ R.T.
# CHECK: the two transformed tensors agree, including their shapes.

## 5. Discover the independent rotational pieces

The nine matrix entries do not all behave as one indivisible object. Discover the smaller families before naming them.

### Your explanation

Split a generic 3×3 matrix into:

1. its trace part;
2. its antisymmetric part;
3. its symmetric-traceless part.

Count their independent numbers and explain, from the rotation experiment, what it means for each family to remain closed under rotation. Use the concrete statement `9 = 1 + 3 + 5` before introducing any angular-order notation.

### Naming checkpoint

Only after the experiment, write your own definition of an **irreducible representation**. Test the phrase *an independent rotational block*: its components may mix with one another, but rotation does not leak them into another block. Where is this analogy useful, and where might it be misleading?

### Blog paragraph

In [ ]:
# LAB 5 — discover 9 = 1 + 3 + 5
# TODO (you): split a generic 3×3 tensor into trace, antisymmetric, and
# symmetric-traceless pieces.
# TODO (you): count the independent values in each piece.
# TODO (you): reconstruct the original tensor from the three pieces.
# TODO (you): compare rotate-then-decompose with decompose-then-rotate.
# CHECK: dimensions are 1 + 3 + 5 = 9 and both rotation routes agree.
# CHECK: trace never leaks into the other pieces, and vice versa.

## 6. Representations and irreducible representations

### Your explanation

Now attach formal names to the experiment. Explain what a representation `D(R)` acts on, why the same physical rotation uses a 1×1, 3×3, or 5×5 matrix on the three blocks above, and why `D(R₂R₁) = D(R₂)D(R₁)` is the feature-space version of your earlier rotation-composition check.

### ML bridge

Relate block structure to keeping different feature types in separate tensor blocks. A representation is the rule for how a feature vector changes; an irrep is a block that cannot be split into smaller rotation-closed blocks.

### Blog paragraph

The next cell is your original first draft. Preserve it, but revisit it only after completing Labs 4–6; annotate what changed in your understanding rather than silently replacing the draft.

The D matrix is the Wagner D matrix for different orders of irreducible order. These are tensors of different size, so they require different rotational tensors. For example, for L = 0, that's a scalar rotation, which makes no difference at all, so it's just a scalar, 1, scaling by 1 for a vector. You usually have a 2-by-2 matrix or, depending on the number of dimensions, a 3-by-3 matrix. For higher-order tensors, you have higher-order matrices. You have these angular breakdowns, etc. 

In [33]:
# LAB 6 — first representations
# TODO (you): represent the same rotation on a scalar and a 3D vector.
# TODO (you): record the shapes of D⁰(R), D¹(R), and D²(R).
# TODO (you): connect those blocks to the 1, 3, and 5 pieces in Lab 5.
# CHECK: D(R₂R₁) = D(R₂)D(R₁) for every feature type.

## 7. Angular orders `l` and components `m`

### Your explanation

Name the three blocks you already discovered as `l = 0`, `l = 1`, and `l = 2`. Explain `l` and `m` without assuming quantum mechanics. Cover the component count `2l + 1`, allowed `m` values, why the components mix under rotation, and why `(l_ξ, m_ξ)` denotes a pair belonging to factor `ξ` rather than equal values.

Create a table for `l = 0, 1, 2, 3` containing component count, allowed `m`, and geometric intuition.

### Blog paragraph

In [34]:
# LAB 7 — angular index structure
# TODO (you): generate valid m values for several l values.
# TODO (you): draw a feature-layout diagram separating channel k from component m.
# CHECK: every l block contains exactly 2l + 1 components.

## 8. Spherical harmonics — Fourier-like modes on the sphere

### Your explanation

Introduce spherical harmonics as fixed direction-dependent basis functions and as a Fourier-like basis for functions on the sphere. Explain their input/output, the visual role of `l` and `m`, why distance is absent, why all `m` values for one `l` form a rotating block, and which real-basis normalization you use.

Use the `l = 1` real harmonics as the first bridge: check numerically whether they are proportional to the unit-direction components `x`, `y`, and `z` under your chosen ordering and normalization. Do not memorize the ordering; document it.

### ML bridge

Compare them with a fixed feature map or positional encoding while explaining the extra rotational structure. Then compare `l` with Fourier frequency and `m` with the components inside one frequency block; record both the power and the limits of this analogy.

### Blog paragraph

In [36]:
# LAB 8A — graph angular orders
# TODO (you): sample directions on the unit sphere.
# TODO (you): evaluate a documented real spherical-harmonic basis for l = 0, 1, 2.
# TODO (you): plot every m component with clear labels and a signed-value legend.
# TODO (you): for l = 1, compare the basis values with normalized x, y, z.
# CHECK: the number of plots for each l equals 2l + 1.

In [37]:
# LAB 8B — basis checks
# TODO (you): estimate the Gram matrix of the sampled basis functions.
# TODO (you): compare it with the result expected from your normalization convention.
# CHECK: different basis functions are approximately orthogonal.

## 9. Wigner-D matrices

### Your explanation

Use the plots above to motivate Wigner-D. Explain what rotates physically, what rotates in feature space, why `Dˡ(R)` has shape `(2l + 1) × (2l + 1)`, and what the identity `Yˡ(Rr̂) = Dˡ(R)Yˡ(r̂)` claims.

### Blog paragraph

In [ ]:
# LAB 9A — inspect Wigner-D actions
# TODO (you): construct D⁰, D¹, and D² using the same basis conventions as Lab 8.
# TODO (you): print shapes and visualize how one feature block changes under rotation.
# CHECK: every Dˡ has shape (2l + 1, 2l + 1).

In [ ]:
# LAB 9B — numerical equivariance
# TODO (you): compare Yˡ(Rr̂) with Dˡ(R)Yˡ(r̂) for random directions.
# TODO (you): test Dˡ(R₁R₂) = Dˡ(R₁)Dˡ(R₂).
# TODO (you): report absolute and relative errors for every l.
# CHECK: both identities hold within your declared tolerance.

## 10. Feature channels versus angular components

### Your explanation

Distinguish channel `k`, angular order `l`, and component `m`. Explain `W[k, k̃, l]` as a learned channel map whose action is shared across every `m` component of one `l` block.

### ML bridge

Connect this to an ordinary dense layer with a symmetry-required tied axis.

### Blog paragraph

In [ ]:
# LAB 10 — channel mixing
# TODO (you): create several channels of one fixed l block.
# TODO (you): implement channel mixing shared across m and test equivariance.
# TODO (you): deliberately use m-dependent weights and repeat the test.
# TODO (you): plot or tabulate the valid and broken equivariance errors.

## 11. Tensor products — return to the outer product

### Your explanation

Return to your two 3D vectors from Lab 4. Distinguish scalar, elementwise, and tensor/outer products. Explain why the raw tensor product has nine components, how tensor product differs from direct sum, why the product is equivariant, and why the nine-dimensional result is reducible.

### ML bridge

Relate this to multiplicative feature interactions while preserving transformation structure.

### Blog paragraph

In [ ]:
# LAB 11 — raw tensor product, now in representation language
# TODO (you): reuse the outer product and rotation check from Lab 4.
# TODO (you): annotate its input and output representations.
# TODO (you): explain why the 9-value result is equivariant but reducible.
# CHECK: the representation-language prediction matches the earlier matrix check.

## 12. Clebsch–Gordan decomposition

### Geometric intuition first

Explain how the 3×3 outer product separates into three rotationally distinct pieces. Connect their component counts to `1 ⊗ 1 → 0 ⊕ 1 ⊕ 2`.

### Coefficients second

Explain CG coefficients as a fixed, precomputed change of basis that combines raw `(m₁, m₂)` components into clean `(L, M)` blocks. State explicitly that the raw product already exists before the coefficients are applied. Record your basis and normalization convention.

### Fourier bridge from your Claude discussion

Compare multiplying spherical modes with multiplying Fourier modes: products create new allowed frequencies/orders, and a change of basis reorganizes them. Then derive the selection rule `m₁ + m₂ = M` from phase addition in the complex basis. Mark clearly which part is analogy and which part is an exact rule.

### Blog paragraph

In [ ]:
# LAB 12A — concrete 1 ⊗ 1 decomposition
# TODO (you): decompose a 3×3 product into its three rotational sectors.
# TODO (you): count each sector and reconstruct the raw product.
# CHECK: dimensions add as 1 + 3 + 5 = 9.
# CHECK: reconstruction matches the original product.

In [ ]:
# LAB 12B — explicit CG coupling
# TODO (you): construct or transcribe the tiny coefficient set for 1 ⊗ 1.
# TODO (you): write explicit loops over m₁ and m₂ before using matrix notation.
# TODO (you): identify which entries survive the m₁ + m₂ = M rule.
# TODO (you): compare coupled blocks with the geometric decomposition above.
# TODO (you): verify each output block transforms with Dᴸ.

## 13. Higher-order coupling: `ν` and `η_ν`

### Your explanation

Using three factors, explain what `ν` counts, how tensor-product order relates to physical body order, why intermediate angular orders appear, and why `η_ν` is needed to distinguish multiple valid coupling paths to the same final `L`. Draw at least two coupling trees.

### Blog paragraph

In [ ]:
# LAB 13 — coupling paths
# TODO (you): enumerate valid intermediate orders for one small ν = 3 example.
# TODO (you): assign η₃ labels and record intermediate/final shapes.
# CHECK: every path obeys the angular-momentum selection rules.

## 14. Radial information

### Your explanation

Explain distance versus direction, fixed Bessel basis functions, the smooth cutoff, learned combinations of fixed basis values, and why radial outputs are rotationally invariant.

### ML bridge

Compare fixed radial expansion followed by learned mixing with a fixed feature map feeding an MLP.

### Blog paragraph

In [ ]:
# LAB 14 — radial basis and cutoff
# TODO (you): graph several Bessel-style basis functions over the cutoff range.
# TODO (you): graph the cutoff and the enveloped basis functions.
# TODO (you): graph one learned linear combination of fixed basis values.
# CHECK: every enveloped basis function smoothly reaches zero at the cutoff.

# Part II — assemble MACE

The remaining sections should feel like composition. Before each equation, list tensors and shapes, name their symmetry type, expand the operations into loops, and connect every operation to a completed lab.

## 15. Eq. 8 — construct `A`

Paste Eq. 8 here. Then write:

- your tensor and shape ledger;
- your inside-out loop expansion;
- your mapping from each operation to a fundamentals lab;
- your explanation for a non-GDL ML practitioner;
- the expected output transformation law.

Your explanation:

In [ ]:
# MACE LAB A — Eq. 8 toy construction
# TODO (you): implement the smallest nontrivial A-feature calculation.
# TODO (you): expose radial, angular, channel-mixed, and coupled intermediates.
# TODO (you): pool over a toy neighborhood.
# CHECK: output shape and equivariance match your written contracts.

## 16. Eq. 9 — first-layer message

Copy Eq. 8 and Eq. 9 above this cell, then derive Eq. 9 line by line.

Your derivation:

- What is special about the first-layer node feature?
- Which indices disappear, and why?
- What happens to the Clebsch–Gordan coefficient?
- What does the remaining neighbor sum compute?

Blog paragraph:

In [ ]:
# MACE LAB B — Eq. 9 as a special case of Eq. 8
# TODO (you): evaluate your Eq. 8 toy calculation with a scalar element feature.
# TODO (you): evaluate the directly simplified Eq. 9 expression on the same inputs.
# CHECK: both routes agree within tolerance.

### Paper-notation checkpoint — generalized CG indices

Map the concrete CG lab above onto the paper's generalized coefficient notation.

Identify the input tuple, output `(L, M)`, and coupling-path label `η_ν`. Record which quantities are indices, fixed coefficients, and learned parameters.

Your explanation:

Your paper-notation translation:

In [ ]:
# PAPER CHECKPOINT — generalized CG notation
# TODO (you): annotate one small generalized coefficient tensor with axis meanings.
# TODO (you): connect one η_ν path to the coupling tree drawn in Lab 11.
# CHECK: every axis and convention is documented before Eq. 10 is implemented.

## 17. Eq. 10 — read it from the inside out

Paste Eq. 10 here. Then write the equivalent loop structure in your own words.

Your explanation of the innermost channel sum:

Your explanation of the product over `ξ`:

Your explanation of the outer angular sum:

Your ν = 2 expansion:

Your blog paragraph:

In [ ]:
# MACE LAB C — Eq. 10 with ν = 2
# TODO (you): mix channels independently for both A factors.
# TODO (you): form the raw product and couple it into selected (L, M) outputs.
# TODO (you): keep the η_ν path axis explicit.
# CHECK: all shapes match the ledger and each B block transforms with Dᴸ.

## 18. Eq. 11 — form the message

Paste Eq. 11 here. Explain which `B` features are combined, which indices disappear, which survive, why the receiving atom's element can affect the weights, and why combining paths with the same output `L` preserves equivariance. Connect the result back to Eq. 7.

Your loop expansion:

Your shape ledger:

Your explanation for a non-GDL ML practitioner:

In [ ]:
# MACE LAB D — Eq. 11 toy message
# TODO (you): combine toy B features across ν and η_ν with learned coefficients.
# TODO (you): preserve the requested output (L, M) block.
# CHECK: the message has the declared shape and transformation law.

## Minimal MACE implementation ladder

Begin only after the equation labs are clear:

- [ ] atomic types and positions
- [ ] cutoff neighbor graph
- [ ] relative distance and direction
- [ ] radial basis and cutoff
- [ ] spherical-harmonic edge features
- [ ] first-layer `A` features
- [ ] `ν = 2` product basis
- [ ] message linear combination
- [ ] node update and residual connection
- [ ] invariant scalar readout
- [ ] total energy as an atom-wise sum
- [ ] forces as the negative position gradient

Before implementing any item, write its input shape, output shape, symmetry contract, and smallest decisive test.

## 19. Global MACE shape ledger — write this yourself

Record the shape and meaning of every axis for:

- `A[i, k, l, m]`
- `w[k, k̃, l]`
- `C[η, L, M, (l₁,m₁), ...]`
- `B[i, η, k, L, M]`

Your shape ledger:


In [ ]:
# GLOBAL SHAPE CHECKS
# TODO (you): assert every shape documented above.
# TODO (you): use deliberately different atom, edge, channel, and component counts
# so that an axis swap cannot pass accidentally.

## 20. Final symmetry checks

Before coding, write the two routes you will compare:

- transform input → run operation
- run operation → transform output

Your explanation of what should match:

Your expected failure modes:

In [ ]:
# FINAL VERIFICATION MATRIX
# TODO (you): translation invariance of relative geometry and scalar energy.
# TODO (you): Wigner-D rotation equivariance of every l block.
# TODO (you): inversion/parity under the declared convention.
# TODO (you): atom-permutation equivariance and total-energy invariance.
# TODO (you): position gradients with the same shape as positions.
# TODO (you): report dtype, absolute error, relative error, and tolerance.

# Blog assembly

Turn the completed notebook into a coherent article for an ML practitioner who is new to GDL.

## Reader promise

What should the reader understand by the end?

## Narrative arc

1. Why ordinary scalar features are insufficient for 3D geometry.
2. How vectors build a nine-component Cartesian tensor.
3. How `9 = 1 + 3 + 5` reveals independent rotational blocks.
4. How representations, irreps, `l`, and `m` name those observations.
5. How spherical harmonics encode direction as Fourier-like modes on the sphere.
6. How Wigner-D matrices express equivariance inside each block.
7. How tensor products create interactions and CG coefficients reorganize them.
8. How MACE assembles the pieces efficiently.

## Editing checklist

- [ ] Every symbol is defined before use.
- [ ] Every abstract term has one concrete example.
- [ ] Every important tensor has a shape.
- [ ] Every transformation claim has a numerical check.
- [ ] Every figure says what the reader should notice.
- [ ] Basis and normalization conventions are explicit.
- [ ] Paper notation appears only after the building block is familiar.

## Your draft

# Pass 3 session log

Append one dated entry after each meaningful session.

## YYYY-MM-DD — topic

**Objective and scope:**

**Reasoning / derivation:**

**Exact cells or artifacts completed:**

**Verification and tolerances:**

**Gotchas / unresolved questions:**

**Next action:**